In [ ]:
import time

import fitz

import fitz  # PyMuPDF

import fitz
from langchain_core.documents import Document

def load_pdf(uploaded_file):
    file_bytes = uploaded_file.read()
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    
    documents = []
    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        if text.strip():
            documents.append(
                Document(
                    page_content=text,
                    metadata={"page": page_num + 1, "source": uploaded_file.name}
                )
            )
    
    doc.close()
    return documents
    
def load_document(filepath):
    with open(filepath, "r")as f:
        content=f.read()
        return content
    


import re
from langchain_core.documents import Document

def chunk_by_paragraph(documents):
    chunks = []

    for doc in documents:
        # Split on single newlines instead, then filter/rejoin small fragments
        lines = doc.page_content.split("\n")
        
        paragraphs = []
        current = ""
        for line in lines:
            line = line.strip()
            if not line:
                continue
            current += " " + line
            if len(current) > 300:  # once we hit a reasonable paragraph size, close it off
                paragraphs.append(current.strip())
                current = ""
        if current.strip():
            paragraphs.append(current.strip())

        for para in paragraphs:
            if len(para) > 40:
                chunks.append(Document(page_content=para, metadata=doc.metadata))

    return chunks

    

class VoyageEmbeddingFunction:
    def __init__(self):
        import voyageai
        import os
        from dotenv import load_dotenv
        load_dotenv()
        self.client = voyageai.Client(api_key=os.getenv("VOYAGE_API_KEY"))

    def __call__(self, input):
        result = self.client.embed(input, model="voyage-3", input_type="document")
        return result.embeddings
    def name(self):
        return "voyage-3"
    def embed_documents(self, input):
        return self.__call__(input)

    def embed_query(self, input):
        result = self.client.embed(input, model="voyage-3", input_type="query")
        return result.embeddings

    
def build_vector_store(chunks, collection_name="policy_docs"):
    import chromadb
    import time

    client = chromadb.PersistentClient(path="chromadb")

    try:
        client.delete_collection(collection_name)
    except Exception:
        pass

    embedding_function = VoyageEmbeddingFunction()
    collection = client.create_collection(collection_name, embedding_function=embedding_function)

    batch_size = 20  # smaller batches, since paragraphs create more chunks
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]

        ids = [f"chunk_{i+j}" for j in range(len(batch))]
        documents = [doc.page_content for doc in batch]
        metadatas = [doc.metadata for doc in batch]
        # retry loop
        while True:
            try:
                collection.add(ids=ids, documents=documents, metadatas=metadatas)
                print(f"Added batch {i // batch_size + 1}")
                break  # success, exit retry loop
            except Exception as e:
                print(f"Rate limit hit, waiting 30 seconds and retrying...")
                time.sleep(30)

        time.sleep(15)  # normal pause between successful batches

    return collection

    
def retrieve(query, collection_name="policy_docs", n_results=2):
    import chromadb
    client = chromadb.PersistentClient(path="chromadb")
    embedding_function = VoyageEmbeddingFunction()
    collection = client.get_collection(collection_name, embedding_function=embedding_function)
    results = collection.query(query_texts=[query], n_results=n_results)
    return results


   



def compare_requirement(requirement, policy_text):
    import os
    from groq import Groq
    from dotenv import load_dotenv

    load_dotenv()

    client = Groq(api_key=os.getenv("GROQ_API_KEY"))

    prompt = f"""
You are a legal compliance analyst.

Evaluate ONLY the requirement below.

Requirement:
{requirement}

Company Policy Evidence:
{policy_text}

Instructions:
- Use ONLY the provided policy evidence.
- Do not assume information that is not present.
- If the evidence is insufficient, say "No matching evidence found."
- Return your analysis in this format:

Status: (Compliant / Partial / Non-Compliant)

Evidence:
<quote from the policy>

Reason:
<why>

Confidence:
<number between 0 and 100>

"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_policy",
            "description": "Search the company's policy documents for sections relevant to a topic or question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The topic or question to search for in the policy documents"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_regulations",
            "description": "Search the web for current laws, regulations, or compliance requirements on a given topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The regulation or legal topic to search for, e.g. 'GDPR data consent requirements'"
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_compliance",
            "description": "Check whether the company's policy on a given topic complies with current regulations. Use this when the user wants a full compliance check, not just a lookup.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The compliance topic to check, e.g. 'GDPR data consent requirements'"
                    }
                },
                "required": ["topic"]
            }
        }
    }
]

def check_compliance(topic):
    # Step 1: Search the regulation
    regulation = search_regulations(
        "general " + topic +
        " under India's Digital Personal Data Protection Act 2023, not sector-specific"
    )

    print("Regulation retrieved.\n")

    # Step 2: Extract individual requirements
    requirements = extract_requirements(regulation)

    if not requirements:
        return "No compliance requirements could be extracted."

    print(f"Extracted {len(requirements)} requirements.\n")

    report = []

    # Step 3: Check each requirement separately
    for req in requirements:
        print("=" * 60)
        print("Checking:", req["topic"])

        query = req["topic"] + " " + " ".join(req["keywords"])
        results = retrieve(query, n_results=3)
        policy_text = "\n\n".join(results["documents"][0])

        analysis = compare_requirement(req["requirement"], policy_text)

        report.append({
            "topic": req["topic"],
            "requirement": req["requirement"],
            "analysis": analysis
        })
        time.sleep(25) 

    # Step 4: Build final report
    final_report = "# Compliance Report\n\n"

    for item in report:
        final_report += f"## {item['topic']}\n"
        final_report += f"Requirement:\n{item['requirement']}\n\n"
        final_report += item["analysis"]
        final_report += "\n\n"
        final_report += "-" * 70
        final_report += "\n\n"

    return final_report

    



def search_regulations(topic):
    import time
    import os
    from tavily import TavilyClient
    from dotenv import load_dotenv

    load_dotenv()
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    response = client.search(
        query=topic, 
        max_results=3,
       include_domains=["meity.gov.in", "dpdpa.com"]
    )

    summary = ""
    for r in response['results']:
        summary += f"Title: {r['title']}\nContent: {r['content']}\n url:{r['url']}"
    return summary

import os
import json
from dotenv import load_dotenv
from groq import Groq

def extract_requirements(regulation_text):
    load_dotenv()

    client = Groq(api_key=os.getenv("GROQ_API_KEY"))

    prompt = f"""
You are a legal compliance analyst.

Extract every explicit compliance requirement from the regulation below.

Rules:
- Return ONLY valid JSON.
- Do NOT include markdown or explanations.
- Ignore introductions and examples.
- Each object must contain:
    - id
    - topic
    - requirement
    - keywords

Example:

[
  {{
    "id": "REQ-1",
    "topic": "Data Retention",
    "requirement": "Personal data shall be deleted when no longer necessary.",
    "keywords": [
      "data retention",
      "delete data",
      "retention period"
    ]
  }}
]

REGULATION:

{regulation_text}
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )


        result = response.choices[0].message.content.strip()

        if result.startswith("```"):
            result = result.replace("```json", "")
            result = result.replace("```", "")
            result = result.strip()

        try:
            requirements = json.loads(result)
            return requirements

        except json.JSONDecodeError:
            print("Model did not return valid JSON.")
            print(result)
            return []
    except Exception as e:
        print("Requirement extraction failed:", e)
        return []


def ask_agent(user_message):
    import os
    from groq import Groq
    from dotenv import load_dotenv
    import time
    load_dotenv()
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": user_message}],
            tools=tools
        )
    except Exception as e:
        print("Agent failed to respond:", e)
        return
    
    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        import json
        args= json.loads(tool_call.function.arguments)
        print("Parsed arguments:", args)
        if tool_call.function.name=="retrieve_policy":
            results=retrieve(args["query"])
            print(results["documents"])
        elif tool_call.function.name=="search_regulations":
            results=search_regulations(args["topic"])
            print(results)
        elif tool_call.function.name=="check_compliance":
            results=check_compliance(args["topic"])
            print(results)
    else:
        print("Model answered directly:", message.content)
    
if __name__ == "__main__":
    documents = load_pdf("data/zomato2.pdf")
    print("loaded pdf")

    chunks = chunk_by_paragraph(documents)
    print("chunks", len(chunks))

    lengths = [len(c.page_content) for c in chunks]
    print("Shortest chunk:", min(lengths), "characters")
    print("Longest chunk:", max(lengths), "characters")
    print("Average chunk length:", sum(lengths) / len(lengths))

    print("\n--- First 3 chunks preview ---")
    for c in chunks[:3]:
        print(f"[{len(c.page_content)} chars] {c.page_content[:150]}...")
        print("---")

    collection = build_vector_store(chunks)
    print("stored", collection.count(), "chunks in vector db")
    total_chunks = collection.count()

    ask_agent("Does Zomato's data retention policy comply with India's DPDPA requirements?")

loaded pdf
chunks 152
Shortest chunk: 45 characters
Longest chunk: 390 characters
Average chunk length: 320.74342105263156

--- First 3 chunks preview ---
[380 chars] Privacy Policy Privacy Policy Last updated on July 25, 2025. Applicability and Scope Eternal Limited (Formerly known as Zomato Limited) and/or its aff...
---
[373 chars] its websites, applications and other online services (collectively, referred as “Services”); and its practices for collecting, using, maintaining, pro...
---
[352 chars] This policy does not apply to information that you provide to, or that is collected by, any third-party, such as restaurants at which you make reserva...
---
Added batch 1
Added batch 2
Added batch 3
Rate limit hit, waiting 30 seconds and retrying...
Added batch 4
Added batch 5
Added batch 6
Rate limit hit, waiting 30 seconds and retrying...
Added batch 7
Added batch 8
stored 152 chunks in vector db
Parsed arguments: {'topic': "Zomato's data retention policy compliance with India's DPDPA 